In [27]:
import argparse
from datetime import datetime, timezone
import pandas as pd

from aind_data_schema.components.identifiers import Code, DataAsset
from aind_data_schema.core.processing import (
    DataProcess,
    Processing,
    ProcessName,
    ProcessStage,
    ResourceTimestamped,
    ResourceUsage,
)
from aind_data_schema_models.units import MemoryUnit
from aind_data_schema_models.system_architecture import OperatingSystem, CPUArchitecture
from codeocean import CodeOcean
import os
os.sys.path.append('/root/capsule/code/beh_ephys_analysis')
from utils.beh_functions import parseSessionID
import json

### Generating sphys spatial crosscorr data asssets' procesing metadata

In [28]:
client = CodeOcean(domain="https://codeocean.allenneuraldynamics.org", token=os.getenv("API_SECRET"))

In [29]:
meta_data_file = '/root/capsule/code/data_management/meta_data_processing.xlsx'
crosscorr_meta = pd.read_excel(meta_data_file, sheet_name='np_spatial_crosscorr')

computation_params_file = '/root/capsule/code/data_management/processing_params.json'
with open(computation_params_file, 'r') as f:
    computation_params = json.load(f)

In [30]:
crosscorr_data_asset_id = 'f9eb95d3-2300-4294-a738-ae4d1cd7016b'
ts = client.data_assets.get_data_asset(data_asset_id=crosscorr_data_asset_id).created
t = datetime.fromtimestamp(ts, tz=timezone.utc)

In [31]:
data_file = '/root/capsule/code/data_management/session_assets.csv'
data_asset_names = ['raw_data', 'sorted_curated']
data_asset_ids = pd.read_csv(data_file)[data_asset_names].dropna().values.flatten().tolist()
data_asset_ids = [id for id in data_asset_ids if len(id) == 36] # filter out any invalid IDs
data_asset_names_full = [client.data_assets.get_data_asset(data_asset_id=id).name  for id in data_asset_ids]
data_assets = [DataAsset(url=name) for name in data_asset_names_full]

In [36]:
dp_list = []
for row_ind, row in crosscorr_meta.iterrows():
    if row['name'] in computation_params:
        params = computation_params[row['name']]
        # print(f"Using parameters for {row['name']}: {params}")
    else:
        params = {}
    curr_code = Code(
        url=row['url'],
        run_script=row['run_script'],
        version='v1.0',
        input_data=data_assets,
        parameters=params,
    )

    curr_dp = DataProcess(
                process_type=ProcessName.OTHER,
                name=row['name'],
                experimenters=["Sue Su"],
                stage=ProcessStage.ANALYSIS,
                start_date_time=t,
                output_path=row['output'],
                code=curr_code,
                notes=row['name'] + '. ' + row['additional_note'] if pd.notna(row['additional_note']) else row['name'],
                )
    dp_list.append(curr_dp)
p_crosscorr_meta = Processing.create_with_sequential_process_graph(
    data_processes=dp_list)

In [38]:
p_crosscorr_meta.data_processes[0].code.parameters

_GenericModel(data_type='curated', probe_type=2)